# Cancer Tumor Classification 
Version 2: PyTorch <n>
Date: 10/25/2026

In [4]:
import logging
import copy

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sb

import torch 
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.svm import SVC
from sklearn.metrics import classification_report, roc_auc_score

## 1. Data Ingestion
Load data from scikit-learn's Wisconsin Breast Cancer dataset

In [5]:
data = load_breast_cancer()
X = data.data
y = data.target
feature_names = data.feature_names

df = pd.DataFrame(X, columns=feature_names)

# Map numeric to descriptive labels
df['Diagnosis'] = pd.Series(y).map({0: 'Malignant', 1: 'Benign'})

logging.info(f"Data sucessfully ingested. Dimensions: {df.shape}")

## 2. EDA

In [11]:
missing = df.isnull().sum()
print(f"Missing values per column: {missing}" )
print()

# Check class distribution
class_counts = df['Diagnosis'].value_counts()
class_percentages = df['Diagnosis'].value_counts(normalize=True)*100

print("Class Distribution:")
for i, label in enumerate(class_counts.index): 
    count = class_counts.values[i]
    percentage = class_percentages.values[i]
    print(f"{label}: {count} samples ({percentage:.2f}%)")
print()

# Scale profiling
numeric_features = df.drop(columns=['Diagnosis'])
display(numeric_features.describe().T[['mean', 'std', 'min', 'max']])

Missing values per column: mean radius                0
mean texture               0
mean perimeter             0
mean area                  0
mean smoothness            0
mean compactness           0
mean concavity             0
mean concave points        0
mean symmetry              0
mean fractal dimension     0
radius error               0
texture error              0
perimeter error            0
area error                 0
smoothness error           0
compactness error          0
concavity error            0
concave points error       0
symmetry error             0
fractal dimension error    0
worst radius               0
worst texture              0
worst perimeter            0
worst area                 0
worst smoothness           0
worst compactness          0
worst concavity            0
worst concave points       0
worst symmetry             0
worst fractal dimension    0
Diagnosis                  0
dtype: int64

Class Distribution:
Benign: 357 samples (62.74%)
Malignant: 

,mean,std,min,max
mean radius,14.127292,3.524049,6.981000,28.11000
mean texture,19.289649,4.301036,9.710000,39.28000
mean perimeter,91.969033,24.298981,43.790000,188.50000
mean area,654.889104,351.914129,143.500000,2501.00000
mean smoothness,0.096360,0.014064,0.052630,0.16340
mean compactness,0.104341,0.052813,0.019380,0.34540
mean concavity,0.088799,0.079720,0.000000,0.42680
mean concave points,0.048919,0.038803,0.000000,0.20120
mean symmetry,0.181162,0.027414,0.106000,0.30400
mean fractal dimension,0.062798,0.007060,0.049960,0.09744


The classes are moderately imbalanced at 63% benign and 37% malignant, so every split is stratified. From the scale profling, we can see that feature scales differ by orders of magnitude, therefore, standardization must happen before PCA.

## 3. Data Standardization & Feature Engineering
The data will be spilt using a 80:20 train-test ratio. The training group will be spit again 80:20 to create a validation set. The validation set is used only for early stopping, and the test set is used for the final evaluation.